In [ ]:
%load_ext autoreload
%autoreload 2
import waffles
import numpy as np
import plotly.subplots as psu
import os
from tqdm import tqdm

from waffles.data_classes.Waveform import Waveform
from waffles.data_classes.WaveformSet import WaveformSet
from waffles.data_classes.ChannelWsGrid import ChannelWsGrid
from waffles.utils.selector_waveforms import WaveformSelector
from waffles.np02_utils.AutoMap import ordered_modules_cathode, ordered_modules_membrane, ordered_modules_pmt, dict_module_to_uniqch, dict_uniqch_to_module, strUch
from waffles.np02_utils.PlotUtils import runBasicWfAnaNP02, wfset_remove_bad_baselines, plot_detectors, plot_averages, plot_mpv_waveforms
from waffles.np02_utils.load_utils import open_processed, ch_read_calib
from waffles.utils.numerical_utils import average_wf_ch

In [ ]:
dettype = "membrane"
dettype = "cathode"
#dettype = "pmt"

datadir = f"/eos/experiment/neutplatform/protodune/experiments/ProtoDUNE-VD/commissioning/"
endpoint = 106 if dettype == "cathode" else 107 if dettype == "membrane" else 110
listmods = ordered_modules_cathode if endpoint == 106 else ordered_modules_membrane if endpoint == 107 else ordered_modules_pmt

n = len(listmods)
group1 = listmods[:n//4]
group2 = listmods[n//4:n//2]
group3 = listmods[n//2:2*(n//3+1)]
group4 = listmods[2*(n//3+1):]

groupLow = group1+group2
groupHig = group3+group4
groupall = group1+group2+group3+group4

try:
    tmpep = list(wfset_full.get_set_of_endpoints())
    if endpoint != tmpep[0]: # recreate the wfset, only one endpoint at a time
        del wfset_full
        wfset_full = None
except:
    wfset_full=None

In [ ]:
run_to_module = {

    "membrane" : {
        40801 : ["M3(1)", "M3(2)"], # Mask 8, int 2250
        40989 : ["M4(1)", "M4(2)"], # Mask 8, int 1900 
        40808 : ["M6(1)", "M6(2)"], # Mask 16, int 4000
        42320 : ["M5(1)", "M5(2)"], # Mask 2, int 4000   
        43229 : ["M7(1)", "M7(2)"], # Mask 4, int 1400 - if it doesn't work try 43230
        42321 : ["M8(1)", "M8(2)"], # Mask 4, int 4000
    },
    "cathode" : { 
       42228 : ["C1(1)", "C1(2)"], # Mask 8, int 4000 - fibers swapped - provare mask 4
       40808 : ["C4(1)", "C4(2)", "C5(1)", "C5(2)", "C7(1)", "C7(2)", "C8(1)", "C8(2)"], # Mask 16, int 4000
       41519 : ["C2(1)", "C2(2)"], # Mask 8, int 3000 
       41536 : ["C3(1)", "C3(2)"], # Mask 1, 2800 
       40807 : ["C6(1)", "C6(2)"], # Mask 8, int 4000  
    },

    "pmt" : {
      43282 : ordered_modules_pmt 
        
    }

}
run_to_module = run_to_module[dettype]

run_to_unich = { r: [ dict_module_to_uniqch[m].channel for m in modules ] for r, modules in run_to_module.items() }
channels = [ x for v in run_to_unich.values() for x in v]

In [ ]:
import copy
import time
def select_channels(waveform: Waveform, channels: list) -> bool:
    if waveform.channel not in channels:
        return False
    if np.any(waveform.adcs[0:500] > 16000):
        return False
    if np.any(waveform.adcs[0:500] < 100 ):
        return False
    return True
def remove_channel_when_run_change(waveform: Waveform, channels_to_remove = []) -> bool:
    if waveform.channel in channels_to_remove:
        return False
    return True
    
def create_wfset(run_to_unich, endpoint, wfset_alread_there=None):

    nwaveforms = None
    wfset_full:WaveformSet = wfset_alread_there 

    if wfset_full is not None:
        for run in wfset_full.runs:
            if run not in run_to_unich.keys():
                if len(wfset_full.runs) != 1:
                    print(f"Removing unused run {run}")
                    wfset_full = WaveformSet.from_filtered_WaveformSet(wfset_full, remove_channel_when_run_change, wfset_full.available_channels[run][endpoint], show_progress=True)
                else:
                    wfset_full = None

    for run, channels in run_to_unich.items():
        if wfset_full is not None:
            redo = True
            
            if run in wfset_full.runs:
                if list(wfset_full.available_channels[run][endpoint]) == sorted(channels):
                    print(f"Run {run} already there :)")
                    redo=False
                
            if redo: 
                try:
                    wfset_full = WaveformSet.from_filtered_WaveformSet(wfset_full, remove_channel_when_run_change, channels_to_remove=channels, show_progress=True)
                except:
                    wfset_full = None
            else:
                continue
                
        wfset = open_processed(run, dettype, datadir, channels, [endpoint], nwaveforms=nwaveforms, verbose=True, mergefiles=False)
        # wfset = WaveformSet.from_filtered_WaveformSet(wfset, select_channels, channels)
        if wfset_full is None:
            wfset_full = copy.deepcopy(wfset)
        else:
            wfset_full.merge(copy.deepcopy(wfset))
        print(f"Loaded run {run}")
    return wfset_full
    
start = time.time()
wfset_full = create_wfset(run_to_unich, endpoint, wfset_full)
end = time.time()
print(end - start)

In [ ]:
runBasicWfAnaNP02(wfset_full, int_ll=250, int_ul=280, amp_ll=100, amp_ul=260, threshold=25, onlyoptimal=False, configyaml="")

In [ ]:
# wfset_optimal = wfset_remove_bad_baselines(wfset_full)
# wfset_optimal
wfset_optimal = wfset_full

In [ ]:
argsheat = dict(
    mode="heatmap",
    analysis_label="std",
    adc_range_above_baseline=2000,
    adc_range_below_baseline=-150,
    adc_bins=125,
    time_bins=wfset_full.points_per_wf//2,
    filtering=16,
    share_y_scale=True,
    share_x_scale=True,
    wfs_per_axes=5000,
    zlog=True,
    width=1550,
    higth=1300,
    showplots=True
)

#detector = groupLow
#detector = groupHig
detector = groupall

#detector=["C1(1)","C1(2)"]

plot_detectors(wfset_optimal, detector, **argsheat)


In [ ]:
cutyaml = 'cuts.yaml'
extractor = WaveformSelector(cutyaml)
wfset_clean = WaveformSet.from_filtered_WaveformSet(wfset_optimal, extractor.applycuts, show_progress=True)
print(f"Original waveforms: {len(wfset_optimal.waveforms)}, after cut: {len(wfset_clean.waveforms)}")

In [ ]:
argsheat['adc_range_above_baseline']=2000
argsheat['adc_range_below_baseline']=-200
argsheat['adc_bins']=125
argsheat['share_y_scale']=True
argsheat['filtering'] = 16
argsheat['return_fig'] = False
plot_detectors(wfset_clean, detector, **argsheat)

In [ ]:
argsheat['adc_range_above_baseline']=2000
argsheat['adc_range_below_baseline']=-200
argsheat['adc_bins']=125
argsheat['return_fig'] = True
figs = plot_detectors(wfset_clean, detector, **argsheat)
fig, rows, cols, title, g = figs[0]
plot_averages(fig, g)
fig.show()

In [ ]:
calibration_file = 'np02-config-v4.0.0.csv'
calibration_data = ch_read_calib(calibration_file)

In [ ]:
from pathlib import Path
template_outputdir = waffles.__path__[0] + "/np02_data/templates/templates_even_larger/"
Path(template_outputdir).mkdir(parents=True, exist_ok=True)

In [ ]:
wfsetch = ChannelWsGrid.clusterize_waveform_set(wfset_clean)[endpoint]

In [ ]:
peaks = {}
for _, ch in np.ndenumerate(g.ch_map.data):
    if ch.channel not in g.ch_wf_sets[ch.endpoint]:
        continue
    wfch = g.ch_wf_sets[ch.endpoint][ch.channel]
    avg = average_wf_ch(wfch)
    peak = np.max(avg)
    peaks[(ch.endpoint,ch.channel)] = peak
    #print(peak)

In [ ]:
#template_type = "MPV"
template_type = "average"
print(f"The template type will be: {template_type}")

In [ ]:
save_template = False

In [ ]:
import glob
# Build reverse lookup: channel integer → module name formatted as C6_1
dict_ch_to_label = {}
for ch in wfsetch.keys():
    module_name = dict_uniqch_to_module[strUch(endpoint, ch)]
    label = module_name.replace("(", "_").replace(")", "")
    dict_ch_to_label[ch] = label

# Delete stale templates only if a DIFFERENT run exists for the same channel
removed = []

if save_template:
    for run_num, run_channels in run_to_unich.items():
        for ch in run_channels:
            module = dict_uniqch_to_module[strUch(endpoint, ch)]
            if module in detector:
                if ch not in wfsetch:
                    continue
                ch_label = dict_ch_to_label.get(ch, str(ch))
                stale = glob.glob(template_outputdir + f"template_*_{ch_label}.txt")
                for old_file in stale:
                    # Extract run number from filename e.g. "template_40807_C6_1.txt" → 40807
                    old_run = int(os.path.basename(old_file).split("_")[1])
                    if old_run != run_num: # only delete if different run
                        os.remove(old_file)
                        removed.append(os.path.basename(old_file))
    
    for f in sorted(removed, key=lambda x: x.split("_", 2)[2]):
        print(f"Removed stale template: {f}")

fig_simple = psu.make_subplots(rows=g.ch_map.rows, cols=g.ch_map.columns)
if template_type == "average":
    plot_averages(fig_simple, g, calibration_data, save=save_template, save_dir=template_outputdir)
    #plot_averages(fig_simple, g)
if template_type == "MPV":
    plot_mpv_waveforms(fig_simple, g, calibration_data, save=save_template, save_dir=template_outputdir, maxfev=20000)

fig_simple.show()

In [ ]:
if save_template:
    with open(template_outputdir+"cuts_used.yaml", "w") as f:
            with open(cutyaml, "r") as fcuts:
                f.write(fcuts.read())

In [ ]:
info_file = template_outputdir + f"wfdetails_{dettype}.info"

existing_data = {}

if os.path.exists(info_file):
    with open(info_file, "r") as f:
        lines = f.readlines()
        for line in lines[1:]:  # Skip header line
            if line.strip():
                # Split by comma to get the main parts
                parts = line.split(", ")
                if len(parts) >= 4:
                    # The first part contains "Module endpoint-channel"
                    module_and_endpointch = parts[0].strip()
                    # Split to separate module from endpoint-channel
                    split_parts = module_and_endpointch.rsplit(" ", 1)
                    if len(split_parts) == 2:
                        module = split_parts[0]
                        endpoint_ch = split_parts[1]
                        run_num = int(parts[1].strip())
                        waveforms = int(parts[2].strip())
                        peak_ch = float(parts[3].strip())
                        existing_data[(endpoint_ch, run_num)] = (module, waveforms, peak_ch)

# Update only channels that match the detector filter
for run_num, run_channels in run_to_unich.items():
    for ch in run_channels:
        module = dict_uniqch_to_module[strUch(endpoint, ch)]
        if module in detector:  # Only update if this module is in the detector list
            wfs = wfsetch[ch]
            endpoint_ch = f"{endpoint}-{ch}"
            peak_ch = peaks[(endpoint,ch)]
            existing_data[(endpoint_ch, run_num)] = (module, len(wfs.waveforms), peak_ch)
            
# Write back to file
with open(info_file, "w") as f:
    f.write("Module, endpoint-channel, run number, # of waveforms, amplitude(ADC)\n")
    for (endpoint_ch, run_num), (module, waveforms, peak_ch) in sorted(existing_data.items(), key=lambda x: x[1][0]):
        f.write(f"{module} {endpoint_ch}, {run_num}, {waveforms}, {peak_ch:.1f}\n")